# Week 10: Reproducible ML Workflow Studio

**Monday 9 November 2026, 14:00–17:00 · NCKU + LAiMM**

The goal is a handover another Python process can validate and replay from saved files. Week 9 diagnosed error mechanisms and froze a controlled comparison; this week starts a new interpreter and working directory. The installed packages still remain shared. This is computational replay of frozen observations, not a new scientific experiment.

## Use one complete notebook

1. Save your own copy (Colab: File → Save a copy in Drive). Choose a free CPU runtime. No GPU, API key, paid AI tool or remote dataset download is needed.
2. Run Setup, then Tasks **1 → 8**. Every task has a prediction cell, runnable scaffold, explanation cell and stopping check. Keep the same notebook for the whole class. The worked alternative is available if you need sample responses; you do not need to run a second copy.
3. H1: Tasks 1–2, 15 practice minutes. H2: Tasks 3–5, 45. H3: Tasks 6–8, 40 (including reflection/export and final summary), plus a 20-minute HW9 → HW10 bridge. Total: 60 minutes explanation + 120 practice.
4. A runtime reset removes variables. Restart from Setup and run in order. Setup recreates ONLY this notebook’s `artifacts/week10-reference/handover` output folder; save edited files elsewhere before restarting.
5. Download `week10-workflow-handover.zip` from the Colab Files sidebar (folder icon → artifacts → week10-reference → ZIP menu → Download). In Jupyter use the final file link. Save your edited .ipynb separately.

**Prerequisites:** stable row IDs, train/validation/test, MAE/RMSE, basic Python arrays. The retained scikit-learn packaged diabetes fixture is used only as a statistical regression example: 442 source rows, ten raw numeric inputs. We do not infer medical, causal or deployment validity. Source-row IDs belong to this frozen dataset version and are not global patient identities.

Tasks 1–8 and the bridge are formative in-class practice. The designated HW10 is a separate individual, graded Homework due 13 November at 23:59 Asia/Taipei. It uses your completed HW9 package, not the classroom ZIP. Written/oral examinations are unchanged. The next meeting on 16 November is the written and oral midterm examination.


**Student exercise:** write your own predictions and explanations.

## Setup

Run once. Missing free libraries are installed only if absent; existing packages are not upgraded. The cell records actual versions and declares candidates, split sizes and tolerances before fitting.

In [ ]:
import os,sys,subprocess,importlib.util,platform,json,csv,hashlib,zipfile,tempfile,shutil
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'1')
required={'numpy':'numpy','matplotlib':'matplotlib','sklearn':'scikit-learn'}
missing=[p for m,p in required.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib,matplotlib.pyplot as plt
import sklearn,scipy
from sklearn.datasets import load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
CONFIG={'seed':2026,'dataset':'sklearn diabetes raw; statistical teaching only',
 'train_n':265,'validation_n':88,'test_n':89,'alphas':[1.,20.,100.],
 'candidate_order':['median','ridge-1','ridge-20','ridge-100'],
 'selection':'validation MAE; exact tie follows candidate_order','atol':1e-8,'rtol':1e-10}
ENV={'python':platform.python_version(),'numpy':np.__version__,'scikit_learn':sklearn.__version__,'matplotlib':matplotlib.__version__,'scipy':scipy.__version__,'platform':platform.platform()}
RUN_LABEL='reference'
ARTIFACTS=Path('artifacts')/('week10-'+RUN_LABEL);ARTIFACTS.mkdir(parents=True,exist_ok=True)
BUNDLE_DIR=ARTIFACTS/'handover'
if BUNDLE_DIR.exists():shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir()
plt.rcParams.update({'figure.figsize':(10,6.9),'savefig.dpi':160,'font.size':22,
 'axes.labelsize':22,'axes.titlesize':24,'xtick.labelsize':17,'ytick.labelsize':18,
 'legend.fontsize':17,'axes.spines.top':False,'axes.spines.right':False})
TEAL='#167b87';RED='#7f2635';GOLD='#b88a29'
def finish(fig,name):
 fig.savefig(ARTIFACTS/name,bbox_inches='tight',facecolor='white');plt.show();plt.close(fig)
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def metrics(y,p):return {'n':int(len(y)),'MAE':float(mean_absolute_error(y,p)),'RMSE':float(np.sqrt(mean_squared_error(y,p)))}
def candidate(name):
 if name=='median':return DummyRegressor(strategy='median')
 return make_pipeline(StandardScaler(),Ridge(alpha=float(name.split('-')[1]),solver='svd'))
print('Actual environment:',ENV);print('Declared configuration:',CONFIG)
responses={};observations={};challenge_observations={}
from IPython.display import display,FileLink


## Task 1: Recover observations by ID (7 min)

**Purpose:** Distinguish row membership, row order and a random seed.

**Do:** Use a 20-ID paper-sized example. A seeded permutation saves 15 training IDs. Reverse the source array and reuse positions: identify the changed membership. Recover rows using the SAVED ID sequence, not just a set filter. No model or 442-row dataset is needed yet.

**Stopping check:** Membership is recovered and the final ID sequence is identical. A seed alone does not bind positions to observations.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task1_prediction"] = ''

In [ ]:
toy_ids=np.array([f'row-{i:02d}' for i in range(20)])
positions=np.random.default_rng(2026).permutation(20)[:15]
saved_ids=toy_ids[positions].tolist(); reversed_ids=toy_ids[::-1]
wrong_ids=reversed_ids[positions].tolist(); lookup={row_id:row_id for row_id in reversed_ids}
recovered=[lookup[row_id] for row_id in saved_ids]
assert recovered==saved_ids
observations['task1']={'saved':saved_ids,'wrong':wrong_ids,'recovered':recovered,
    'positional_membership_changed':set(wrong_ids)!=set(saved_ids),'same_recovered_sequence':recovered==saved_ids}
print(json.dumps(observations['task1'],indent=2))

In [ ]:
responses["task1_explanation"] = ''

## Task 2: Freeze raw data and check preprocessing scope (8 min)

**Purpose:** Make fitted preprocessing belong only to training rows.

**Do:** Load the retained 442-row packaged regression fixture with scaled=False. Create source IDs before splitting; freeze CSV and 265/88/89 membership. Check disjointness and the full union. Fit a diagnostic scaler on training rows ONLY, then compare its means with training means. The all-row mean gap is descriptive, not a second competing leaky model.

**Stopping check:** 442 unique IDs; 265/88/89 disjoint exhaustive membership; training means match the diagnostic scaler. No clinical interpretation is required.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task2_prediction"] = ''

In [ ]:
data=load_diabetes(scaled=False)
X=np.asarray(data.data,dtype=float);y=np.asarray(data.target,dtype=float)
FEATURES=list(data.feature_names);IDS=np.array([f'diabetes-{i:04d}' for i in range(len(y))])
assert X.shape==(442,10) and y.shape==(442,)
assert np.isfinite(X).all() and np.isfinite(y).all()
order=np.random.default_rng(CONFIG['seed']).permutation(len(y))
train_idx=np.sort(order[:265]);val_idx=np.sort(order[265:353]);test_idx=np.sort(order[353:])
SPLITS={k:IDS[idx].tolist() for k,idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]}
assert all(not(set(SPLITS[a])&set(SPLITS[b])) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert set(sum(SPLITS.values(),[]))==set(IDS)
Xtr,ytr=X[train_idx],y[train_idx];Xva,yva=X[val_idx],y[val_idx];Xte,yte=X[test_idx],y[test_idx]
HEADER=['row_id',*FEATURES,'target']
DATA_PATH=BUNDLE_DIR/'frozen.csv'
with DATA_PATH.open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(HEADER)
 for i in range(len(y)):w.writerow([IDS[i],*[repr(float(v)) for v in X[i]],repr(float(y[i]))])
(BUNDLE_DIR/'split.json').write_text(json.dumps(SPLITS,indent=2),encoding='utf-8')
data_sha256=digest(DATA_PATH)
print('Rows / features:',X.shape,'Columns:',HEADER)
print('Partition sizes:',{k:len(v) for k,v in SPLITS.items()});print('CSV SHA-256:',data_sha256)

# Show training targets only; final-test outcomes are first displayed in Task 5.
fig,ax=plt.subplots(layout='constrained');ax.hist(ytr,bins=14,color=TEAL)
ax.set(xlabel='Training target only',ylabel='Count',title='Training cohort; validation/test outcomes not displayed here')
finish(fig,'w10-target-distributions.png')

scope_scaler=StandardScaler().fit(Xtr)
np.testing.assert_allclose(scope_scaler.mean_,Xtr.mean(axis=0),atol=1e-12)
mean_gap=(X.mean(axis=0)-Xtr.mean(axis=0))/scope_scaler.scale_
observations['task2']={'counts':{k:len(v) for k,v in SPLITS.items()},'training_mean_matches':True,
 'mean_gap_train_sd':mean_gap.tolist(),'row_id_is_feature':False}
fig,ax=plt.subplots(layout='constrained');ax.bar(FEATURES,mean_gap,color=RED);ax.axhline(0,color='black')
ax.set(xlabel='Input coordinate',ylabel='(All-row mean - train mean) / train SD');finish(fig,'w10-preprocessing-scope.png')
print(observations['task2'])

In [ ]:
responses["task2_explanation"] = ''

## Task 3: Fit candidates and freeze the validation choice (15 min)

**Purpose:** Separate training loss, model selection and final evaluation.

**Do:** Fit the training-target median and Ridge alpha 1, 20, 100 pipelines using the SAME training rows. Ridge minimizes squared error plus an L2 penalty; selection uses validation MAE. The listed order breaks exact ties. Save the choice now; final-test outcomes appear only in Task 5. Inspect the actual coefficient-norm curve separately from validation MAE.

**Stopping check:** Four training-only candidates; one saved validation choice; scaler means agree with Task 2. The median baseline remains eligible to win.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task3_prediction"] = ''

In [ ]:
models={name:candidate(name).fit(Xtr,ytr) for name in CONFIG['candidate_order']}
validation={name:metrics(yva,m.predict(Xva)) for name,m in models.items()}
selected=min(CONFIG['candidate_order'],key=lambda name:(validation[name]['MAE'],CONFIG['candidate_order'].index(name)))
print('Validation:',json.dumps(validation,indent=2));print('Selected before final test:',selected)
fig,ax=plt.subplots(layout='constrained')
ax.bar(range(4),[validation[n]['MAE'] for n in models],color=[TEAL if n==selected else RED for n in models])
ax.set_xticks(range(4),['Median','Ridge 1','Ridge 20','Ridge 100']);ax.set_ylabel('Validation MAE (target units)');finish(fig,'w10-validation-selection.png')
fig,ax=plt.subplots(layout='constrained');alphas=CONFIG['alphas']
norms=[float(np.linalg.norm(models['ridge-'+f'{a:g}'].named_steps['ridge'].coef_)) for a in alphas]
ax.plot(alphas,norms,'o-',color=TEAL,lw=2);ax.set_xscale('log');ax.set(xlabel='Ridge alpha',ylabel='Standardized coefficient L2 norm');finish(fig,'w10-regularization-path.png')
for name,m in models.items():
 if name!='median':np.testing.assert_allclose(m.named_steps['standardscaler'].mean_,scope_scaler.mean_)
selected_before_test=selected
observations['task3']={'validation':validation,'selected':selected,'test_used_for_selection':False}

In [ ]:
responses["task3_explanation"] = ''

## Review the portable replay program

Expand the code cell below and review its named steps: verify hashes → read the data contract → reconstruct saved ID cohorts → read reference predictions → refit the frozen candidate → compare aligned vectors. This cell writes the provided source and imports its functions; it does not train a model. Task 4 calls only the schema reader. Task 5 executes a copied file in a separate process. Only run reviewed, trusted code; a checksum is not a signature.

In [ ]:
RUNNER = r'''
"""Reviewed Week10 replay program. Frozen files in, aligned comparison out."""
import argparse,csv,json,hashlib,os,platform
from pathlib import Path
import numpy as np
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
PAYLOAD={'frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json'}
def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def read_contract(root,config):
    root=Path(root)
    with (root/'frozen.csv').open(newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['row_id',*config['features'],'target']:raise ValueError('schema: exact column order required')
        rows=list(reader)
    if len(rows)!=config['row_count']:raise ValueError('schema: row count')
    if any(None in r or any(v is None for v in r.values()) for r in rows):raise ValueError('schema: row width')
    ids=[r['row_id'] for r in rows]
    if any(not i.strip() for i in ids):raise ValueError('schema: empty row ID')
    if len(set(ids))!=len(ids):raise ValueError('schema: duplicate row ID')
    values=np.array([[float(r[k]) for k in [*config['features'],'target']] for r in rows])
    if not np.isfinite(values).all():raise ValueError('schema: non-finite numeric value')
    splits=json.loads((root/'split.json').read_text())
    if set(splits)!={'train','validation','test'}:raise ValueError('schema: split names')
    if not all(isinstance(v,list) and all(isinstance(i,str) for i in v) for v in splits.values()):raise ValueError('schema: ID lists required')
    flat=sum(splits.values(),[])
    if len(flat)!=len(set(flat)) or set(flat)!=set(ids):raise ValueError('schema: split overlap or missing IDs')
    if {k:len(v) for k,v in splits.items()}!=config['split_counts']:raise ValueError('schema: split counts')
    lookup={r['row_id']:r for r in rows}
    def cohort(name):
        subset=[lookup[i] for i in splits[name]]
        return np.array([[float(r[k]) for k in config['features']] for r in subset]),np.array([float(r['target']) for r in subset])
    return cohort,splits

def run(root,out):
    root=Path(root).resolve();manifest=json.loads((root/'manifest.json').read_text())
    if manifest.get('format')!='MLES-W10-v2' or set(manifest['sha256'])!=PAYLOAD:raise ValueError('manifest: exact required file list and format')
    for name,value in manifest['sha256'].items():
        if digest(root/name)!=value:raise ValueError('checksum mismatch: '+name)
    config=json.loads((root/'config.json').read_text());cohort,splits=read_contract(root,config)
    Xtr,ytr=cohort('train');Xte,yte=cohort('test')
    chosen=config['selected'];allowed=['median','ridge-1','ridge-20','ridge-100']
    if config['candidate_order']!=allowed or chosen not in allowed:raise ValueError('config: undeclared candidate')
    # The runner itself reads and validates reference predictions; no notebook globals are needed.
    with (root/'reference-predictions.csv').open(newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['row_id','observed','predicted']:raise ValueError('reference: exact header required')
        reference=list(reader)
    if any(None in r or any(v is None for v in r.values()) for r in reference):raise ValueError('reference: row width')
    if [r['row_id'] for r in reference]!=splits['test']:raise ValueError('reference: test ID sequence mismatch')
    ref_y=np.array([float(r['observed']) for r in reference]);ref_p=np.array([float(r['predicted']) for r in reference])
    if not np.isfinite(ref_y).all() or not np.isfinite(ref_p).all():raise ValueError('reference: non-finite value')
    np.testing.assert_array_equal(ref_y,yte)
    model=DummyRegressor(strategy='median') if chosen=='median' else make_pipeline(StandardScaler(),Ridge(alpha=float(chosen.split('-')[1]),solver='svd'))
    model.fit(Xtr,ytr);pred=model.predict(Xte)
    np.testing.assert_allclose(pred,ref_p,atol=config['atol'],rtol=config['rtol'])
    error=pred-yte;delta=pred-ref_p
    result={'selected':chosen,'data_sha256':digest(root/'frozen.csv'),'test_ids':splits['test'],'predictions':pred.tolist(),
        'metrics':{'n':len(yte),'MAE':float(np.abs(error).mean()),'RMSE':float(np.sqrt((error**2).mean()))},
        'reference_comparison_passed':True,'max_reference_difference':float(np.max(np.abs(delta))),
        'process_id':os.getpid(),'working_directory_name':Path.cwd().name,
        'environment':{'python':platform.python_version(),'numpy':np.__version__,'scikit_learn':sklearn.__version__},
        'scope':'Fresh interpreter and working directory, same installed environment; not independent scientific replication'}
    Path(out).write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf-8')
    print(json.dumps({'selected':chosen,'test_n':len(yte),'reference_comparison_passed':True}))
    return result
if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--out',required=True);a=p.parse_args();run(a.root,a.out)

'''
(BUNDLE_DIR/'replay.py').write_text(RUNNER,encoding='utf-8')
import importlib.util
spec=importlib.util.spec_from_file_location('reviewed_w10_runner',BUNDLE_DIR/'replay.py')
runner_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(runner_module)
print('Reviewed replay source written; no fitting performed by this cell.')

## Task 4: Reject malformed copies before fitting (12 min)

**Purpose:** Give the data interface executable stopping conditions.

**Do:** Expand and review the provided replay program before this task. Work on temporary copies. Predict the expected rejection for duplicate IDs, NaN, wrong headers, row width, empty IDs and an invalid partition. Call the schema reader directly, so this test checks input contracts without training. Keep the original CSV hash unchanged.

**Stopping check:** Every declared bad copy is rejected with a recorded reason. The original reference bytes remain unchanged.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task4_prediction"] = ''

In [ ]:
FROZEN_CONFIG={**CONFIG,'features':FEATURES,'row_count':442,'split_counts':{'train':265,'validation':88,'test':89},'selected':selected,'environment':ENV}
def contract_probe(kind):
 with tempfile.TemporaryDirectory(prefix='mles-w10-probe-') as work:
  target=Path(work);shutil.copy2(DATA_PATH,target/'frozen.csv');shutil.copy2(BUNDLE_DIR/'split.json',target/'split.json')
  with (target/'frozen.csv').open(newline='',encoding='utf-8') as f:rows=list(csv.reader(f))
  if kind=='duplicate ID':rows[2][0]=rows[1][0]
  elif kind=='non-finite':rows[1][1]='nan'
  elif kind=='column order':rows=[[r[1],r[0],*r[2:]] for r in rows]
  elif kind=='extra value':rows[1].append('unexpected')
  elif kind=='missing value':rows[1].pop()
  elif kind=='empty ID':rows[1][0]=''
  elif kind=='invalid partition':
   bad=json.loads((target/'split.json').read_text());bad['test'][0]=bad['train'][0];(target/'split.json').write_text(json.dumps(bad))
  else:raise ValueError(kind)
  with (target/'frozen.csv').open('w',newline='',encoding='utf-8') as f:csv.writer(f).writerows(rows)
  try:runner_module.read_contract(target,FROZEN_CONFIG)
  except ValueError as error:return {'case':kind,'rejected':True,'reason':str(error)}
  raise AssertionError('Malformed copy unexpectedly accepted: '+kind)
contract_checks=[contract_probe(k) for k in ['duplicate ID','non-finite','column order','extra value','missing value','empty ID','invalid partition']]
assert digest(DATA_PATH)==data_sha256
observations['task4']=contract_checks
print(json.dumps(contract_checks,indent=2))
fig,ax=plt.subplots(layout='constrained');ax.barh([r['case'] for r in contract_checks],[1]*len(contract_checks),color=TEAL)
ax.set(xlim=(0,1.2),xlabel='Actual rejection outcome');ax.set_xticks([0,1],['Accepted','Rejected']);finish(fig,'w10-contract-tests.png')

In [ ]:
responses["task4_explanation"] = ''

## Task 5: Evaluate once and replay from copied files (18 min)

**Purpose:** Build a handover that can check its own predictions without notebook variables.

**Do:** Open final-test outcomes after the validation choice is frozen. Compare the selected model with the declared median baseline; inspect one actual failure ID. Build six payload files and a manifest. Copy the bundle into a new temporary directory and invoke its replay.py in a NEW Python interpreter. The runner must read the reference-predictions CSV itself and compare all 89 aligned values. Record a different process ID; do not call the runner function in this notebook and call that a fresh process.

**Stopping check:** All file hashes, schema and exact test-ID sequence pass. The child independently compares predictions within atol=1e-8 and rtol=1e-10. This still shares installed dependencies.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task5_prediction"] = ''

In [ ]:
test_prediction={n:models[n].predict(Xte) for n in dict.fromkeys(['median',selected])}
test_metrics={n:metrics(yte,p) for n,p in test_prediction.items()}
pred=test_prediction[selected];residual=pred-yte
print('Final test:',json.dumps(test_metrics,indent=2))
fig,ax=plt.subplots(layout='constrained');names=list(test_metrics);width=.34;xx=np.arange(len(names))
for j,m in enumerate(['MAE','RMSE']):ax.bar(xx+(j-.5)*width,[test_metrics[n][m] for n in names],width,label=m)
ax.set_xticks(xx,names);ax.set_ylabel('Target units');ax.legend();finish(fig,'w10-test-metrics.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(yte,pred,s=36,alpha=.8,color=TEAL)
limits=[float(min(yte.min(),pred.min())),float(max(yte.max(),pred.max()))];ax.plot(limits,limits,'k--',lw=1)
ax.set(xlabel='Observed test target',ylabel='Selected-model prediction');finish(fig,'w10-test-parity.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(pred,residual,s=36,color=RED,alpha=.8);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Selected-model prediction',ylabel='Prediction - observation');finish(fig,'w10-test-residuals.png')
fig,ax=plt.subplots(layout='constrained')
for n,p in test_prediction.items():
 e=np.sort(np.abs(p-yte));ax.plot(e,np.arange(1,len(e)+1)/len(e),label=n,lw=2)
ax.set(xlabel='Absolute error (target units)',ylabel='Empirical cumulative fraction');ax.legend();finish(fig,'w10-error-cdf.png')
worst=np.argsort(np.abs(residual))[-5:][::-1]
worst_rows=[{'row_id':IDS[test_idx[j]],'observed':float(yte[j]),'predicted':float(pred[j]),'residual':float(residual[j])} for j in worst]
print('Five largest absolute residuals:',json.dumps(worst_rows,indent=2))
assert selected==selected_before_test
FROZEN_CONFIG={**CONFIG,'features':FEATURES,'row_count':442,'split_counts':{'train':265,'validation':88,'test':89},'selected':selected,'environment':ENV}
(BUNDLE_DIR/'config.json').write_text(json.dumps(FROZEN_CONFIG,indent=2),encoding='utf-8')
with (BUNDLE_DIR/'reference-predictions.csv').open('w',newline='',encoding='utf-8') as f:
 w=csv.writer(f);w.writerow(['row_id','observed','predicted'])
 for i,actual,p in zip(IDS[test_idx],yte,pred):w.writerow([i,repr(float(actual)),repr(float(p))])
record={'week':10,'data_scope':'Packaged regression dataset; statistical teaching only, not clinical validation',
 'config':CONFIG,'environment':ENV,'data_sha256':data_sha256,'selected_on_validation':selected,
 'validation':validation,'test':test_metrics,'worst_test_rows':worst_rows,
 'claim_limit':'Frozen random-row teaching split; no external clinical, causal or deployment claim.'}
(BUNDLE_DIR/'result.json').write_text(json.dumps(record,indent=2,allow_nan=False),encoding='utf-8')
BUNDLE_FILES=['frozen.csv','split.json','config.json','reference-predictions.csv','replay.py','result.json']
manifest={'format':'MLES-W10-v2','sha256':{name:digest(BUNDLE_DIR/name) for name in BUNDLE_FILES}}
(BUNDLE_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Handover files:',BUNDLE_FILES+['manifest.json'])
def fresh_process(root,label):
 output=(ARTIFACTS/(label+'-process-result.json')).resolve()
 with tempfile.TemporaryDirectory(prefix='mles-w10-fresh-') as work:
  copied=Path(work)/'handover';shutil.copytree(root,copied)
  proc=subprocess.run([sys.executable,'-I',str(copied/'replay.py'),'--root',str(copied),'--out',str(output)],cwd=work,capture_output=True,text=True,timeout=120)
  if proc.returncode:raise RuntimeError(proc.stderr)
  result=json.loads(output.read_text());assert result['process_id']!=os.getpid()
  assert result['working_directory_name']==Path(work).name
  return result
replay=fresh_process(BUNDLE_DIR,'reference')
assert replay['test_ids']==SPLITS['test'] and replay['reference_comparison_passed']
np.testing.assert_allclose(replay['predictions'],pred,atol=CONFIG['atol'],rtol=CONFIG['rtol'])
replay_diff=np.asarray(replay['predictions'])-pred;max_replay_difference=float(np.max(np.abs(replay_diff)))
observations['task5']={'different_process':True,'same_installed_environment':True,'runner_compares_its_own_reference':True,'max_difference':max_replay_difference,'worst_case':worst_rows[0]}
fig,ax=plt.subplots(layout='constrained');ax.scatter(np.arange(len(pred)),replay_diff,color=TEAL);ax.axhline(0,color='black');ax.set(ylim=(-1e-7,1e-7),xlabel='Saved test-ID position',ylabel='Fresh process - reference prediction');finish(fig,'w10-replay-differences.png')
print(observations['task5'])

In [ ]:
responses["task5_explanation"] = ''

## Task 6: Reorder CSV rows, preserve the records (15 min)

**Purpose:** Distinguish changed bytes from changed records, then test row-order independence.

**Do:** Reverse data rows in a temporary COPY, keep the header and saved split-ID lists. Confirm that the exact CSV hash changes while the entire ID-keyed record mapping stays equal. Record this reviewed transformation and update ONLY that copy’s CSV digest in its manifest. Run a fresh process and compare aligned predictions. The reference bundle stays unchanged. Updating a manifest alone never proves a transformation is legitimate.

**Stopping check:** Different CSV hashes, equal ID-keyed values, equal test-ID order and predictions within tolerance. No new winner is selected.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task6_prediction"] = ''

In [ ]:
with tempfile.TemporaryDirectory(prefix='mles-w10-reorder-') as work:
 copied=Path(work)/'handover';shutil.copytree(BUNDLE_DIR,copied)
 with (copied/'frozen.csv').open(newline='',encoding='utf-8') as f:rows=list(csv.reader(f))
 original_records={row[0]:row[1:] for row in rows[1:]}
 reordered=[rows[0],*rows[:0:-1]]
 assert {row[0]:row[1:] for row in reordered[1:]}==original_records
 with (copied/'frozen.csv').open('w',newline='',encoding='utf-8') as f:csv.writer(f).writerows(reordered)
 new_hash=digest(copied/'frozen.csv');assert new_hash!=data_sha256
 changed_manifest=json.loads((copied/'manifest.json').read_text());changed_manifest['sha256']['frozen.csv']=new_hash
 (copied/'manifest.json').write_text(json.dumps(changed_manifest,indent=2))
 reordered_replay=fresh_process(copied,'reordered')
 assert reordered_replay['test_ids']==SPLITS['test']
 np.testing.assert_allclose(reordered_replay['predictions'],pred,atol=CONFIG['atol'],rtol=CONFIG['rtol'])
observations['task6']={'reference_sha256':data_sha256,'reviewed_copy_sha256':new_hash,'ID_keyed_records_equal':True,'saved_ID_sequence_equal':True,'max_prediction_difference':reordered_replay['max_reference_difference'],'reference_unchanged':digest(DATA_PATH)==data_sha256}
assert observations['task6']['reference_unchanged'];print(observations['task6'])

In [ ]:
responses["task6_explanation"] = ''

## Task 7: Check a coordinate explanation (12 min)

**Purpose:** Explain the fitted function in its declared coordinate system.

**Do:** If the selected candidate is Ridge, inspect it; if the median won, explicitly use the separately declared Ridge-20 interpretation candidate. Recover raw coefficients and intercept, then check all predictions. Increase coordinate bmi by 0.1 TRAINING SD, holding others fixed; compare every prediction delta with 0.1 times its standardized coefficient. This tests a function, not a feasible medical intervention.

**Stopping check:** The raw equation and the entire perturbation vector match within tolerance; the interpretation candidate is recorded.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task7_prediction"] = ''

In [ ]:
explain_name=selected if selected!='median' else 'ridge-20'
explain_model=models[explain_name];scaler=explain_model.named_steps['standardscaler'];reg=explain_model.named_steps['ridge']
raw_coef=reg.coef_/scaler.scale_;raw_intercept=float(reg.intercept_-np.dot(raw_coef,scaler.mean_))
np.testing.assert_allclose(raw_intercept+Xte@raw_coef,explain_model.predict(Xte),atol=1e-8,rtol=1e-10)
coefficient_record={'candidate':explain_name,'features':FEATURES,'standardized':reg.coef_.tolist(),
 'raw':raw_coef.tolist(),'raw_intercept':raw_intercept,'training_mean':scaler.mean_.tolist(),'training_scale':scaler.scale_.tolist()}
fig,ax=plt.subplots(layout='constrained');ax.barh(FEATURES,reg.coef_,color=[TEAL if v>=0 else RED for v in reg.coef_]);ax.axvline(0,color='black',lw=1)
ax.set(xlabel='Response change per +1 training SD',ylabel='Input coordinate',title=explain_name+' / function, not causation');finish(fig,'w10-coefficients.png')
mean_gap=(X.mean(axis=0)-Xtr.mean(axis=0))/Xtr.std(axis=0)
fig,ax=plt.subplots(layout='constrained');ax.bar(FEATURES,mean_gap,color=RED);ax.axhline(0,color='black',lw=1)
ax.set(xlabel='Feature',ylabel='(All-row mean - train mean) / train SD');finish(fig,'w10-preprocessing-scope.png')
j=FEATURES.index('bmi');changed=Xte.copy();changed[:,j]+=.1*scaler.scale_[j]
observed_delta=explain_model.predict(changed)-explain_model.predict(Xte);expected_delta=.1*reg.coef_[j]
np.testing.assert_allclose(observed_delta,np.full(len(yte),expected_delta),atol=CONFIG['atol'],rtol=CONFIG['rtol'])
observations['task7']={'candidate':explain_name,'coordinate':FEATURES[j],'step_training_SD':.1,'expected_delta':float(expected_delta),'max_disagreement':float(np.max(np.abs(observed_delta-expected_delta)))}
print(observations['task7'])

In [ ]:
responses["task7_explanation"] = ''

## Task 8: Audit, explain and hand over (13 min)

**Purpose:** Connect a portable artifact with an actual result and a bounded claim.

**Do:** Verify all six payload hashes again. Show a partner the validation winner, one actual failure row, test MAE and the separate replay difference. Locate the CSV, split, config, reviewed source, reference predictions and result record. Complete the three reflection fields and export below. Keep the eight predictions and explanations tied to your own observed outputs.

**Stopping check:** A partner can locate the evidence without your hidden variables. All 19 fields are checked for completeness; blank fields remain an honest draft.

Fill the prediction before running, then explain one actual output and its boundary.

In [ ]:
responses["task8_prediction"] = ''

In [ ]:
checks={name:digest(BUNDLE_DIR/name)==sha for name,sha in manifest['sha256'].items()}
assert all(checks.values()) and digest(DATA_PATH)==data_sha256
observations['task8']={'payload_hashes':checks,'selected':selected,'test_metric':test_metrics[selected],'one_failure_ID':worst_rows[0]['row_id'],'fresh_process_difference':max_replay_difference}
print(json.dumps(observations['task8'],indent=2))

In [ ]:
responses["task8_explanation"] = ''

## Optional A: tamper detection; Optional B: explicit name adapter

Complete all eight core tasks and the HW9 → HW10 bridge first. The challenges are optional extensions outside the required class route. Set `RUN_CHALLENGES=True` below only if time remains. A changes a copied CSV without updating its manifest and expects rejection before an output is written. B restores the canonical feature order by name and verifies exact values. Both leave the reference unchanged. Neither checksum matching nor a working adapter establishes scientific validity.

In [ ]:

RUN_CHALLENGES=False
challenge_observations={}
if RUN_CHALLENGES:
 # A: unchanged manifest must reject a changed file, even a blank-line-only byte change.
 with tempfile.TemporaryDirectory(prefix='mles-w10-tamper-') as work:
  copied=Path(work)/'handover';shutil.copytree(BUNDLE_DIR,copied)
  (copied/'frozen.csv').write_bytes((copied/'frozen.csv').read_bytes()+b'\n')
  output=Path(work)/'must-not-exist.json'
  proc=subprocess.run([sys.executable,'-I',str(copied/'replay.py'),'--root',str(copied),'--out',str(output)],cwd=work,capture_output=True,text=True,timeout=120)
  assert proc.returncode!=0 and 'checksum mismatch' in proc.stderr and not output.exists()
  challenge_observations['tamper']={'changed_bytes_rejected':True,'no_result_written':True}
 # B: explicit name adapter; do not silently accept a positional mismatch.
 order=list(reversed(FEATURES));wrong=X[:,[FEATURES.index(name) for name in order]]
 restored=np.column_stack([wrong[:,order.index(name)] for name in FEATURES])
 np.testing.assert_array_equal(restored,X)
 challenge_observations['name_adapter']={'strict_rejection':contract_probe('column order')['rejected'],'canonical_values_restored':True}
 assert digest(DATA_PATH)==data_sha256
 print(challenge_observations)
else:print('Optional branches skipped. Eight core tasks are sufficient.')


## Reflect, peer-review and download

Within Task 8, show the saved winner, one actual failure ID, the prediction error against outcomes, and the separate replay difference. Fill the three fields below. Eight predictions + eight explanations + three reflections = **19 fields**. Blanks produce a draft; filled fields only indicate readiness for discussion. Write an accurate AI-use disclosure (`none` if appropriate).

In [ ]:
responses['controlled_change'] = ''
responses['limitation'] = ''
responses['ai_disclosure'] = ''

In [ ]:

keys=[f'task{i}_{kind}' for i in range(1,9) for kind in ['prediction','explanation']]+['controlled_change','limitation','ai_disclosure']
blank=[k for k in keys if not isinstance(responses.get(k),str) or not responses[k].strip()]
status='draft' if blank else 'ready for peer discussion; completeness only, not correctness'
record.update({'replay_max_prediction_difference':max_replay_difference,'contract_tests':contract_checks,'replay_scope':replay['scope'],'observations':observations,'responses':responses,'blank_fields':blank,'response_status':status,'optional':challenge_observations,'coefficients':coefficient_record})
(ARTIFACTS/'week10-evidence.json').write_text(json.dumps(record,indent=2,allow_nan=False))
HANDOVER_ZIP=ARTIFACTS/'week10-workflow-handover.zip'
with zipfile.ZipFile(HANDOVER_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for name in sorted([*BUNDLE_FILES,'manifest.json']):z.write(BUNDLE_DIR/name,arcname='handover/'+name)
 for p in sorted(ARTIFACTS.glob('w10-*.png')):z.write(p,arcname='figures/'+p.name)
 z.write(ARTIFACTS/'week10-evidence.json',arcname='week10-evidence.json')
with zipfile.ZipFile(HANDOVER_ZIP) as z:assert z.testzip() is None
print(status,'Completed fields:',19-len(blank),'/19','Missing:',blank)
print('Download ZIP:',HANDOVER_ZIP,'Save your edited notebook too.')
print('After extraction and source review: python handover/replay.py --root handover --out replay-result.json')
display(FileLink(str(HANDOVER_ZIP)))


## Summary and examination bridge

Preserve observation identity and declared choices. Make schema rejection explicit. Give the replay program enough files to compare its own reference predictions. Distinguish exact bytes from records, and distinguish replay difference from prediction error. Hand over evidence with a bounded conclusion.

**Written-exam practice:** explain why a seed alone is insufficient and compute the tolerance for a reference value of 10. **Oral-exam practice:** use a real row from your run to explain the selected model, one failure and one limitation. This practice does not replace either examination.

## Primary references

- [Raw packaged regression data](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
- [Training-only pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Random stream compatibility](https://numpy.org/doc/stable/reference/random/compatibility.html)
- [Model persistence and trusted code](https://scikit-learn.org/stable/model_persistence.html)

Original documentation checked 27 October 2026; HW9 → HW10 teaching bridge updated 9 November. Classroom exercises are ungraded. HW10 is the already designated Workflow Homework, not a midterm project.

## Bridge: from your HW9 evidence to the HW10 receiver (20 minutes)

**Keep all eight tasks.** Tasks 1–8 use the packaged diabetes fixture; this bridge transfers the same method to the synthetic equipment case. Task 8 includes the original reflection and classroom ZIP export. Use the final 20 studio minutes for this bridge; optional A/B are extensions after the core work, not extra required class time.

| Interface | Tasks 1–8 classroom example | Graded HW9 → HW10 |
|---|---|---|
| Data | 442 statistical-fixture rows; 265/88/89 | 600 synthetic independent equipment runs; 360/120/120 |
| Identity | saved source IDs, ten inputs | run_id, three inputs and declared physical units |
| Receiver | 89 selected-model predictions | 120 baseline **and** candidate predictions |
| Correct package | week10-workflow-handover.zip | your completed hw09-evidence.zip |
| Manifest | handover/manifest.json with sha256 | flat manifest.json with files; assignment HW9 |

**Do not substitute the classroom ZIP for HW9.** The example below creates a labeled teaching package using the actual HW9 exporter and HW10 receiver functions. It is public guided practice, not an independently completed Homework or peer exchange. The real HW10 starter rejects its classroom-demo marker.

**Predict first:** Which files identify the rows, chosen model and expected vector? What should a missing package produce? Record answers before execution. This adds four bridge reflections; the original 19 Task 1–8 fields remain intact.

In [ ]:
bridge_responses = {'package_prediction': '', 'replay_prediction': '', 'rejection_prediction': '', 'completion_explanation': ''}

### Bridge 1 · Inspect and create an HW9-format example (5 min)
Run the setup and sender cells. The fixed CSV and reviewed support code are embedded, so no dataset service or paid API is needed. Inspect `BRIDGE_CONFIG`: the supplied teaching change is alpha 1 → 20 with the same raw inputs. Read the frozen validation choice **before** test metrics. A candidate need not improve to produce a valid package. The two plots are actual outputs from these synthetic observations.

In [ ]:
# A separate directory and module keep Tasks 1-8 unchanged.
from pathlib import Path
import base64, zlib, json, importlib.util, sys, csv, io, zipfile, shutil, hashlib
BRIDGE_DIR = Path('artifacts/week10-hw9-hw10-demo')
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)
BRIDGE_HELPER = '"""Support code for two individual ML Homework assignments; synthetic data only.\nNo network, API key, GPU, pickle or joblib is used. Review submitted Python source\nbefore executing another student\'s package. Integrity hashes are not signatures.\n"""\nfrom pathlib import Path\nimport csv, json, hashlib, io, zipfile, tempfile, shutil, sys, subprocess, platform, html, base64, importlib.util, uuid, contextlib\nimport numpy as np\nimport sklearn, scipy\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom sklearn.tree import DecisionTreeRegressor\nFEATURES=[\'power_W\',\'airflow_m_s\',\'ambient_C\']\nCOLUMNS=[\'run_id\',*FEATURES,\'temp_rise_C\',\'split\']\nUNITS={\'power_W\':\'W\',\'airflow_m_s\':\'m/s\',\'ambient_C\':\'degC\',\'temp_rise_C\':\'degC difference\'}\nBASE_CONFIG={\'learner\':\'ridge\',\'alpha\':1.0,\'max_depth\':3}\nGROUPS={\'low_airflow\':lambda X:X[:,1]<=1.,\'high_power\':lambda X:X[:,0]>=75.,\'hot_ambient\':lambda X:X[:,2]>=28.}\nREPORT_FIELDS=[\'change\',\'prediction\',\'result\',\'evidence\',\'limitations\']\n\ndef digest(data):return hashlib.sha256(data).hexdigest()\ndef jd(data):return json.dumps(data,sort_keys=True,separators=(\',\',\':\'),allow_nan=False)\ndef versions():return {\'python\':platform.python_version(),\'numpy\':np.__version__,\'scipy\':scipy.__version__,\'scikit-learn\':sklearn.__version__}\ndef write_json(path,data):Path(path).write_text(json.dumps(data,indent=2,allow_nan=False),encoding=\'utf8\')\ndef load_table(path,contract):\n    if contract.get(\'units\')!=UNITS:raise ValueError(\'unit contract mismatch; do not silently relabel or convert inputs\')\n    with Path(path).open(newline=\'\',encoding=\'utf8\') as f:\n        reader=csv.DictReader(f)\n        if reader.fieldnames!=COLUMNS:raise ValueError(\'CSV columns or order differ from the declared schema\')\n        rows=list(reader)\n    ids=[r[\'run_id\'] for r in rows]\n    if len(ids)!=len(set(ids)):raise ValueError(\'duplicate run_id\')\n    if len(rows)!=contract[\'rows\']:raise ValueError(\'row count mismatch\')\n    numeric=[]\n    for r in rows:\n        try:v=[float(r[k]) for k in FEATURES+[\'temp_rise_C\']]\n        except (ValueError,TypeError):raise ValueError(\'non-numeric measurement\') from None\n        if not np.all(np.isfinite(v)):raise ValueError(\'non-finite measurement\')\n        if not (0<=v[0]<=200 and .1<=v[1]<=10 and 0<=v[2]<=60):raise ValueError(\'input outside declared encoding bounds\')\n        if r[\'split\'] not in [\'train\',\'validation\',\'test\']:raise ValueError(\'unknown split label\')\n        numeric.append(v)\n    order=np.argsort(ids);rows=[rows[i] for i in order];numeric=np.asarray(numeric)[order];ids=np.array(ids)[order]\n    records=[{\'run_id\':r[\'run_id\'],**{k:float(r[k]) for k in FEATURES+[\'temp_rise_C\']},\'split\':r[\'split\']} for r in rows]\n    if digest(jd(records).encode())!=contract[\'records_sha256\']:raise ValueError(\'ID-keyed records differ from the frozen data contract\')\n    split=np.array([r[\'split\'] for r in rows]);counts={k:int(np.sum(split==k)) for k in [\'train\',\'validation\',\'test\']}\n    if counts!=contract[\'split_counts\']:raise ValueError(\'split counts mismatch\')\n    return {\'ids\':ids,\'X\':numeric[:,:3],\'y\':numeric[:,3],\'split\':split,\'records_sha256\':contract[\'records_sha256\']}\n\ndef features_from_source(source):\n    # This is ordinary reviewed course Python, not a sandbox for hostile code.\n    scope={\'np\':np};exec(compile(source,\'candidate_features.py\',\'exec\'),scope)\n    if not callable(scope.get(\'candidate_features\')):raise ValueError(\'define candidate_features(X)\')\n    return scope[\'candidate_features\']\ndef checked_features(fn,X):\n    before=X.copy();a=np.asarray(fn(X),dtype=float);b=np.asarray(fn(X),dtype=float)\n    if not np.array_equal(before,X):raise ValueError(\'feature function changed X in place\')\n    if a.ndim!=2 or a.shape[0]!=len(X) or a.shape[1]<3:raise ValueError(\'feature shape must be rows by at least three columns\')\n    if not np.array_equal(a[:,:3],X):raise ValueError(\'preserve the three original columns first\')\n    if not np.all(np.isfinite(a)) or not np.array_equal(a,b):raise ValueError(\'features must be finite and deterministic\')\n    return a\n\ndef estimator(config):\n    if set(config)!=set(BASE_CONFIG) or config[\'learner\'] not in [\'ridge\',\'tree\']:raise ValueError(\'unknown model configuration\')\n    alpha=config[\'alpha\'];depth=config[\'max_depth\']\n    if not isinstance(alpha,(int,float)) or isinstance(alpha,bool) or not np.isfinite(alpha) or not (0<alpha<=1000):raise ValueError(\'alpha must be finite in (0,1000]\')\n    if not isinstance(depth,int) or isinstance(depth,bool) or not (1<=depth<=6):raise ValueError(\'tree depth must be an integer from 1 to 6\')\n    learner=Ridge(alpha=alpha,solver=\'svd\') if config[\'learner\']==\'ridge\' else DecisionTreeRegressor(max_depth=depth,min_samples_leaf=15,random_state=2031)\n    return make_pipeline(StandardScaler(),learner)\ndef score(y,p):return {\'MAE\':float(np.mean(np.abs(p-y))),\'RMSE\':float(np.sqrt(np.mean((p-y)**2))),\'n\':len(y)}\ndef comparison(table,source,config,include_test=False):\n    X,y,split=table[\'X\'],table[\'y\'],table[\'split\'];tr=split==\'train\';va=split==\'validation\'\n    fn=features_from_source(source);trans=checked_features(fn,X.copy())\n    if config[\'learner\']==\'ridge\' and config[\'max_depth\']!=BASE_CONFIG[\'max_depth\']:raise ValueError(\'max_depth has no effect on Ridge; change an active setting\')\n    if config[\'learner\']==\'tree\' and config[\'alpha\']!=BASE_CONFIG[\'alpha\']:raise ValueError(\'alpha has no effect on this tree; retain the default unused field\')\n    feature_change=not(trans.shape==X.shape and np.array_equal(trans,X));model_change=config!=BASE_CONFIG\n    if feature_change and model_change:raise ValueError(\'change the feature basis OR model settings, one factor family at a time\')\n    baseline=estimator(BASE_CONFIG).fit(X[tr],y[tr]);candidate=estimator(config).fit(trans[tr],y[tr])\n    result={\'implemented\':feature_change or model_change,\'versions\':versions(),\'validation\':{\'baseline\':score(y[va],baseline.predict(X[va])),\'candidate\':score(y[va],candidate.predict(trans[va]))}}\n    result[\'selected\']=\'candidate\' if result[\'validation\'][\'candidate\'][\'MAE\']<result[\'validation\'][\'baseline\'][\'MAE\'] else \'baseline\'\n    if include_test:\n        te=split==\'test\';pb=baseline.predict(X[te]);pc=candidate.predict(trans[te]);result[\'test\']={\'baseline\':score(y[te],pb),\'candidate\':score(y[te],pc)}\n        result[\'slices\']={}\n        for name,group in GROUPS.items():\n            mask=group(X[te])\n            if mask.any():result[\'slices\'][name]={\'baseline\':score(y[te][mask],pb[mask]),\'candidate\':score(y[te][mask],pc[mask])}\n        result[\'test_ids\']=table[\'ids\'][te].tolist();result[\'predictions\']={\'baseline\':pb.tolist(),\'candidate\':pc.tolist()}\n    return result\n\ndef freeze_plan(path,source,config,plan,table):\n    if plan.get(\'failure_group\') not in GROUPS:raise ValueError(\'choose low_airflow,high_power,or hot_ambient before freezing\')\n    result=comparison(table,source,config)\n    proposal={\'source\':source,\'config\':config,\'plan\':plan,\'records_sha256\':table[\'records_sha256\'],\'selected_on_validation\':result[\'selected\'],\'validation\':result[\'validation\']}\n    proposal[\'proposal_sha256\']=digest(jd(proposal).encode());write_json(path,proposal);return proposal\n\ndef run_frozen(table,frozen,source,config,plan):\n    check={k:v for k,v in frozen.items() if k!=\'proposal_sha256\'}\n    if digest(jd(check).encode())!=frozen[\'proposal_sha256\'] or source!=frozen[\'source\'] or config!=frozen[\'config\'] or plan!=frozen[\'plan\'] or table[\'records_sha256\']!=frozen[\'records_sha256\']:raise ValueError(\'proposal changed after freeze; declare a new plan and disclose any repeated test use\')\n    result=comparison(table,source,config,include_test=True)\n    if result[\'selected\']!=frozen[\'selected_on_validation\']:raise ValueError(\'validation choice differs\')\n    return result\n\ndef one_page(path,title,fields,disclosure):\n    # Five short sections and disclosure; this file is the separately submitted explanation.\n    words=sum(len(str(v).split()) for v in fields.values())+len(str(disclosure).split())\n    if words>450:raise ValueError(\'one-page explanation exceeds 450 words; shorten it\')\n    body=\'<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>\'+html.escape(title)+\'</title><style>@page{size:A4;margin:18mm}body{max-width:760px;margin:20px auto;padding:12px;font:11pt/1.4 Arial;color:#12233f}h1{font-size:18pt}h2{font-size:12pt;margin:14px 0 4px}p{margin:0}footer{margin-top:18px;font-size:9pt;color:#52666b}</style><h1>\'+html.escape(title)+\'</h1>\'\n    for key in REPORT_FIELDS:body+=\'<h2>\'+html.escape(key.title())+\'</h2><p>\'+html.escape(str(fields.get(key,\'\')).strip() or \'[Complete this section]\')+\'</p>\'\n    body+=\'<h2>AI disclosure</h2><p>\'+html.escape(str(disclosure) or \'[State assistance or none]\')+\'</p><footer>NCKU + LAiMM · Individual Homework · Synthetic teaching case · \'+str(words)+\' words</footer></html>\'\n    Path(path).write_text(body,encoding=\'utf8\');return words\n\ndef hw9_export(csv_path,contract,frozen,result,fields,disclosure,out_dir=\'hw09-output\'):\n    table=load_table(csv_path,contract);check=run_frozen(table,frozen,frozen[\'source\'],frozen[\'config\'],frozen[\'plan\'])\n    if jd(check)!=jd(result):raise ValueError(\'result does not match the frozen executable protocol\')\n    status=\'ready for review\' if result[\'implemented\'] and all(str(frozen[\'plan\'].get(k,\'\')).strip() for k in [\'hypothesis\',\'expected\',\'held_fixed\',\'failure_condition\']) and all(str(fields.get(k,\'\')).strip() for k in REPORT_FIELDS) and str(disclosure).strip() else \'draft\'\n    target=Path(out_dir);target.mkdir(exist_ok=True);run=target/(\'run-\'+uuid.uuid4().hex[:10]);run.mkdir();bundle=run/\'bundle\';bundle.mkdir();figdir=bundle/\'figures\';figdir.mkdir()\n    shutil.copy2(csv_path,bundle/\'frozen.csv\');write_json(bundle/\'contract.json\',contract);write_json(bundle/\'experiment.json\',frozen);write_json(bundle/\'result.json\',result);write_json(bundle/\'environment.json\',versions());(bundle/\'candidate_features.py\').write_text(frozen[\'source\']);shutil.copy2(Path(__file__),bundle/\'hw_workflow.py\')\n    with (bundle/\'reference-predictions.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.writer(f);writer.writerow([\'run_id\',\'baseline\',\'candidate\'])\n        writer.writerows(zip(result[\'test_ids\'],result[\'predictions\'][\'baseline\'],result[\'predictions\'][\'candidate\']))\n    replay="from pathlib import Path\\nimport json,sys\\nfrom hw_workflow import replay_directory\\nif __name__==\'__main__\':\\n    result=replay_directory(Path(__file__).parent)\\n    print(json.dumps(result,indent=2))\\n"\n    (bundle/\'replay.py\').write_text(replay)\n    (bundle/\'requirements.txt\').write_text(\'\\n\'.join(k+\'==\'+v for k,v in versions().items() if k!=\'python\')+\'\\n\')\n    (bundle/\'README.txt\').write_text(\'Inspect the Python source first. Use CPU and the recorded package versions. Run: python replay.py\\nThis is a reviewed class package, not a sandbox. A new process is not automatically an independent person or environment.\\nThe receiver compares every ID-aligned baseline and candidate test prediction with atol=1e-8,rtol=1e-10. Do not change the frozen selection from test scores.\\n\')\n    import matplotlib.pyplot as plt\n    X,y,split=table[\'X\'],table[\'y\'],table[\'split\'];te=split==\'test\'\n    fig,axes=plt.subplots(1,2,figsize=(10,4),sharex=True,sharey=True)\n    for ax,kind in zip(axes,[\'baseline\',\'candidate\']):ax.scatter(X[te,1],np.asarray(result[\'predictions\'][kind])-y[te],s=20);ax.axhline(0,c=\'black\',lw=1);ax.set(title=kind,xlabel=\'airflow (m/s)\',ylabel=\'prediction - observation (degC)\')\n    fig.tight_layout();fig.savefig(figdir/\'residuals.png\',dpi=140);plt.show();plt.close(fig)\n    fig,ax=plt.subplots(figsize=(7,4));ax.bar([\'baseline\',\'candidate\'],[result[\'test\'][k][\'MAE\'] for k in [\'baseline\',\'candidate\']]);ax.set(ylabel=\'test MAE (degC)\',title=\'Predeclared diagnosis; selection was frozen on validation\');fig.tight_layout();fig.savefig(figdir/\'comparison.png\',dpi=140);plt.show();plt.close(fig)\n    words=one_page(run/\'HW9-explanation.html\',\'HW9 · Diagnose and defend a change\',fields,disclosure);shutil.copy2(run/\'HW9-explanation.html\',bundle/\'explanation.html\')\n    gallery=\'<!doctype html><html lang="en"><meta charset="utf-8"><title>HW9 evidence</title><style>body{max-width:1000px;margin:auto;font:18px/1.5 sans-serif}img{max-width:100%}pre{white-space:pre-wrap}</style><h1>HW9 evidence: \'+status+\'</h1><pre>\'+html.escape(json.dumps({\'validation\':result[\'validation\'],\'test\':result[\'test\'],\'slices\':result[\'slices\']},indent=2))+\'</pre>\'\n    for file in sorted(figdir.glob(\'*.png\')):gallery+=\'<h2>\'+file.stem+\'</h2><img alt="\'+file.stem+\'" src="data:image/png;base64,\'+base64.b64encode(file.read_bytes()).decode()+\'">\'\n    (bundle/\'evidence-report.html\').write_text(gallery)\n    manifest={\'assignment\':\'HW9\',\'status\':status,\'scope\':\'Synthetic individual experiment; presence is not correctness.\',\'files\':{str(p.relative_to(bundle)):digest(p.read_bytes()) for p in sorted(bundle.rglob(\'*\')) if p.is_file()}}\n    write_json(bundle/\'manifest.json\',manifest)\n    package=run/\'hw09-evidence.zip\'\n    with zipfile.ZipFile(package,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(bundle.rglob(\'*\')):\n            if p.is_file():z.write(p,str(p.relative_to(bundle)))\n    print(\'HW9:\',status,\'\\nEvidence ZIP:\',package,\'\\nSeparate one-page explanation:\',run/\'HW9-explanation.html\',\'\\nWords:\',words)\n    return {\'status\':status,\'zip\':str(package.resolve()),\'explanation\':str((run/\'HW9-explanation.html\').resolve()),\'bundle\':str(bundle.resolve())}\n\ndef validate_manifest(folder):\n    folder=Path(folder);m=json.loads((folder/\'manifest.json\').read_text())\n    for name,h in m[\'files\'].items():\n        p=Path(name)\n        if p.is_absolute() or \'..\' in p.parts:raise ValueError(\'unsafe manifest path\')\n        if not (folder/name).is_file() or digest((folder/name).read_bytes())!=h:raise ValueError(\'file checksum mismatch: \'+name)\n    return m\n\ndef unpack(package,destination):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=False)\n    with zipfile.ZipFile(package) as z:\n        if sum(i.file_size for i in z.infolist())>20_000_000:raise ValueError(\'class package exceeds 20 MB\')\n        for i in z.infolist():\n            p=Path(i.filename)\n            if p.is_absolute() or \'..\' in p.parts or ((i.external_attr>>16)&0o170000)==0o120000:raise ValueError(\'unsafe ZIP path or symlink\')\n        z.extractall(destination)\n    validate_manifest(destination);return destination\n\ndef replay_directory(folder):\n    folder=Path(folder);m=validate_manifest(folder)\n    if m[\'status\']!=\'ready for review\':raise ValueError(\'HW9 package is a draft; finish your actual change and explanation first\')\n    contract=json.loads((folder/\'contract.json\').read_text());table=load_table(folder/\'frozen.csv\',contract);frozen=json.loads((folder/\'experiment.json\').read_text())\n    if (folder/\'candidate_features.py\').read_text()!=frozen[\'source\']:raise ValueError(\'feature source differs from frozen declaration\')\n    result=run_frozen(table,frozen,frozen[\'source\'],frozen[\'config\'],frozen[\'plan\'])\n    with (folder/\'reference-predictions.csv\').open() as f:records=list(csv.DictReader(f))\n    ids=[r[\'run_id\'] for r in records]\n    if len(ids)!=len(set(ids)) or set(ids)!=set(result[\'test_ids\']):raise ValueError(\'reference prediction IDs mismatch\')\n    by_id={r[\'run_id\']:r for r in records};differences={}\n    for name in [\'baseline\',\'candidate\']:\n        expected=np.array([float(by_id[i][name]) for i in result[\'test_ids\']]);actual=np.array(result[\'predictions\'][name])\n        if not np.allclose(actual,expected,rtol=1e-10,atol=1e-8):raise ValueError(\'prediction vector mismatch for \'+name)\n        differences[name]=float(np.max(np.abs(actual-expected)))\n    return {\'status\':\'reproduced\',\'rows_compared\':len(ids),\'max_abs_difference\':differences,\'atol\':1e-8,\'rtol\':1e-10,\'selected_on_validation\':result[\'selected\'],\'receiver_versions\':versions(),\'sender_versions\':json.loads((folder/\'environment.json\').read_text()),\'scope\':\'New Python process in receiver working directory; independence requires a separate human receiver.\'}\n\ndef fresh_process_replay(package,destination):\n    folder=unpack(package,destination)\n    process=subprocess.run([sys.executable,str((folder/\'replay.py\').resolve())],cwd=folder,capture_output=True,text=True,timeout=60)\n    receipt={\'returncode\':process.returncode,\'stdout\':process.stdout,\'stderr\':process.stderr,\'command\':\'python replay.py\',\'fresh_process\':True}\n    if process.returncode==0:receipt[\'result\']=json.loads(process.stdout)\n    return folder,receipt\n\ndef negative_test(folder,mutate_contract):\n    contract=json.loads((Path(folder)/\'contract.json\').read_text());before=jd(contract);bad=json.loads(before);mutate_contract(bad)\n    if jd(bad)==before:return {\'status\':\'not implemented\',\'expected_rejection\':False}\n    try:load_table(Path(folder)/\'frozen.csv\',bad)\n    except ValueError as e:return {\'status\':\'rejected\',\'expected_rejection\':True,\'exception\':type(e).__name__,\'message\':str(e)}\n    return {\'status\':\'unexpectedly accepted\',\'expected_rejection\':False}\n\ndef hw10_export(receipt,rejection,fields,disclosure,verification_mode,receiver_status,history=None,package_path=None,out_dir=\'hw10-output\'):\n    if verification_mode not in [\'peer\',\'self-replay\']:raise ValueError(\'mode must honestly be peer or self-replay\')\n    if verification_mode==\'peer\' and not str(receiver_status).strip():raise ValueError(\'describe the actual independent receiver and action; a mode label is not evidence\')\n    complete=receipt.get(\'returncode\')==0 and rejection.get(\'expected_rejection\') and all(str(fields.get(k,\'\')).strip() for k in REPORT_FIELDS) and str(disclosure).strip() and str(receiver_status).strip()\n    status=(\'ready for review\' if verification_mode==\'peer\' else \'self-replay; instructor/TA verification pending\') if complete else \'draft or unresolved replay failure\'\n    out=Path(out_dir)/(\'run-\'+uuid.uuid4().hex[:10]);out.mkdir(parents=True);write_json(out/\'replay-receipt.json\',receipt);write_json(out/\'negative-test.json\',rejection);write_json(out/\'verification.json\',{\'mode\':verification_mode,\'receiver_status\':receiver_status,\'status\':status,\'AI_disclosure\':disclosure,\'presence_is_not_correctness\':True});write_json(out/\'receiver-environment.json\',versions());one_page(out/\'HW10-explanation.html\',\'HW10 · Handover and reproducibility\',fields,disclosure)\n    write_json(out/\'replay-history.json\',history or [receipt])\n    if package_path is not None and Path(package_path).is_file():write_json(out/\'package-identity.json\',{\'filename\':Path(package_path).name,\'sha256\':digest(Path(package_path).read_bytes())})\n    files=sorted(p for p in out.iterdir() if p.is_file());write_json(out/\'manifest.json\',{\'status\':status,\'files\':{p.name:digest(p.read_bytes()) for p in files}})\n    package=out/\'hw10-evidence.zip\'\n    with zipfile.ZipFile(package,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.iterdir()):\n            if p.is_file() and p!=package:z.write(p,p.name)\n    print(\'HW10:\',status,\'\\nEvidence ZIP:\',package,\'\\nSeparate one-page explanation:\',out/\'HW10-explanation.html\')\n    return {\'status\':status,\'zip\':str(package.resolve()),\'explanation\':str((out/\'HW10-explanation.html\').resolve())}\n'
(BRIDGE_DIR/'hw_workflow.py').write_text(BRIDGE_HELPER, encoding='utf8')
_bridge_spec = importlib.util.spec_from_file_location('w10_thermal_demo', BRIDGE_DIR/'hw_workflow.py')
bridge_hw = importlib.util.module_from_spec(_bridge_spec)
_bridge_spec.loader.exec_module(bridge_hw)
BRIDGE_DATA = zlib.decompress(base64.b64decode('...'))
(BRIDGE_DIR/'thermal-sensor.csv').write_bytes(BRIDGE_DATA)
BRIDGE_CONTRACT = {'case': 'Independent synthetic equipment temperature-rise experiments', 'rows': 600, 'split_counts': {'train': 360, 'validation': 120, 'test': 120}, 'units': {'power_W': 'W', 'airflow_m_s': 'm/s', 'ambient_C': 'degC', 'temp_rise_C': 'degC difference'}, 'records_sha256': '3d197032b92337a88dc630e5313d2be65eef776a35118adec61130b495daaecf', 'source': 'Original course-generated synthetic fixture; no measured plant data or physical validation.', 'split_rationale': 'Independent simulated runs,not a time series. Fixed disjoint row IDs; random IID split is a teaching assumption.', 'encoding_bounds': {'power_W': [0, 200], 'airflow_m_s': [0.1, 10], 'ambient_C': [0, 60]}, 'target': 'Temperature rise above the ambient condition,not absolute temperature.'}
BRIDGE_PLOT_STYLE = {'font.size':11, 'axes.labelsize':11, 'axes.titlesize':14,
    'xtick.labelsize':10, 'ytick.labelsize':10, 'figure.titlesize':13,
    'legend.fontsize':10}
bridge_table = bridge_hw.load_table(BRIDGE_DIR/'thermal-sensor.csv', BRIDGE_CONTRACT)
print('GUIDED CLASSROOM DEMONSTRATION; not a student Homework submission.')
print('Actual runtime:', bridge_hw.versions())
print('Fixed independent synthetic runs:', BRIDGE_CONTRACT['split_counts'])
print('The original Tasks 1-8 diabetes fixture and handover remain unchanged.')


In [ ]:
# A supplied teaching comparison, not a model choice to copy into HW9.
BRIDGE_FEATURE_SOURCE = "def candidate_features(X):\n    return X.copy()\n"
BRIDGE_CONFIG = {'learner':'ridge', 'alpha':20.0, 'max_depth':3}
BRIDGE_PLAN = {
 'hypothesis':'Changing only Ridge alpha changes shrinkage; it may change validation error.',
 'expected':'A smaller coefficient norm is plausible; lower held-out error is not guaranteed.',
 'held_fixed':'Same raw features, 360/120/120 rows, train-only scaling and SVD solver.',
 'failure_group':'low_airflow',
 'failure_condition':'The predeclared slice has airflow at most 1 m/s.'
}
bridge_frozen = bridge_hw.freeze_plan(BRIDGE_DIR/'frozen-proposal.json',
    BRIDGE_FEATURE_SOURCE, BRIDGE_CONFIG, BRIDGE_PLAN, bridge_table)
print('Frozen before test:', bridge_frozen['selected_on_validation'])
bridge_result = bridge_hw.run_frozen(bridge_table, bridge_frozen,
    BRIDGE_FEATURE_SOURCE, BRIDGE_CONFIG, BRIDGE_PLAN)
print(json.dumps({k:bridge_result[k] for k in ['validation','test','selected','implemented']}, indent=2))
_bm = bridge_result['test']['baseline']['MAE']
_cm = bridge_result['test']['candidate']['MAE']
BRIDGE_REPORT = {
 'change': 'This supplied classroom demonstration predicts equipment temperature rise above ambient in degrees Celsius difference. It retains the fixed independent synthetic runs and the original raw-input Ridge baseline. The single candidate edit changes alpha from one to twenty. Features, split membership, scaling scope and solver remain fixed. This is not a student Homework submission.',
 'prediction': 'The prior expectation is that stronger regularization may shrink coefficients, but it need not reduce held-out prediction error. Validation selects between the baseline and this one declared candidate. The choice is saved before test outcomes are displayed. The low-airflow condition is declared before that inspection, rather than discovered by repeatedly searching test results.',
 'result': f'This actual run produced baseline test MAE {_bm:.8f} and candidate test MAE {_cm:.8f} degrees Celsius difference. Both models have 120 test predictions. The validation-selected model is '+bridge_result['selected']+'. A larger candidate error is a valid outcome. We retain these predictions and the observed failures instead of rewriting the frozen comparison to promise improvement.',
 'evidence': 'The exported package contains the exact CSV, units contract, frozen experiment, original candidate source, two ID-aligned reference prediction columns, runtime versions, reviewed replay source and two actual diagnostic charts. The manifest hashes every payload file. A separate receiver interpreter reads those files and compares complete vectors. Its result will be saved separately from the prediction-error analysis.',
 'limitations': 'These are synthetic independent experiments, not validation of real equipment. A changed regularization setting is not a physical explanation. Replaying the same observations does not test generalization to another population. The two processes in this guided demonstration share installed dependencies and are operated by the same person. Neither matching hashes nor a matching prediction vector establishes independent human verification or scientific validity.'
}
BRIDGE_DISCLOSURE = 'Supplied instructor classroom example prepared with AI assistance; actual code, predictions, figures and exports must be checked. Replace this example with your own work for Homework.'
with plt.rc_context(BRIDGE_PLOT_STYLE):
    bridge_author = bridge_hw.hw9_export(BRIDGE_DIR/'thermal-sensor.csv', BRIDGE_CONTRACT,
        bridge_frozen, bridge_result, BRIDGE_REPORT, BRIDGE_DISCLOSURE,
        out_dir=BRIDGE_DIR/'author-runs')
# Label the example in its verified manifest. The real HW10 starter refuses it.
bridge_bundle = Path(bridge_author['bundle'])
bridge_manifest = json.loads((bridge_bundle/'manifest.json').read_text())
bridge_manifest['classroom_demo'] = True
bridge_manifest['purpose'] = 'Guided Week10 example; use your own completed HW9 for graded HW10.'
bridge_hw.write_json(bridge_bundle/'manifest.json', bridge_manifest)
BRIDGE_PACKAGE = BRIDGE_DIR/'demo-hw09-evidence.zip'
with zipfile.ZipFile(BRIDGE_PACKAGE, 'w', zipfile.ZIP_DEFLATED) as _z:
    for _p in sorted(bridge_bundle.rglob('*')):
        if _p.is_file(): _z.write(_p, str(_p.relative_to(bridge_bundle)))
assert bridge_author['status'] == 'ready for review'
print('Created the HW9-format demonstration package:', BRIDGE_PACKAGE)
print('Its classroom_demo marker prevents use as a graded HW10 submission.')


### Bridge 2 · Receive, refit and compare all IDs (5 min)
Read the displayed replay source before running it. The real receiver functions extract to a new folder, validate hashes and schema, refit the frozen comparison and compare both 120-row vectors. The new interpreter shares installed packages; it is not another person. Inspect the complete receipt and measured difference plot. Keep target error and replay difference separate.

In [ ]:
# Inspect the same files HW10 needs, then use its actual receiver functions.
with zipfile.ZipFile(BRIDGE_PACKAGE) as _z:
    _manifest = json.loads(_z.read('manifest.json'))
    print('Assignment:', _manifest['assignment'], '| status:', _manifest['status'])
    print('Example marker:', _manifest['classroom_demo'])
    for _name in ['README.txt','requirements.txt','replay.py','contract.json','experiment.json']:
        assert _name in _z.namelist()
    print('Reviewed replay.py:\n', _z.read('replay.py').decode())
bridge_received, bridge_receipt = bridge_hw.fresh_process_replay(
    BRIDGE_PACKAGE, BRIDGE_DIR/('receiver-'+__import__('uuid').uuid4().hex[:10]))
print(json.dumps(bridge_receipt, indent=2))
assert bridge_receipt['returncode'] == 0, bridge_receipt['stderr']
assert bridge_receipt['result']['rows_compared'] == 120
assert set(bridge_receipt['result']['max_abs_difference']) == {'baseline','candidate'}
# Capture actual receiver vectors in another fresh process for an honest plot.
# The standard HW10 receipt already verifies its own reference but reports maxima.
BRIDGE_CAPTURE_SOURCE = """from pathlib import Path
import json, sys
folder=Path(sys.argv[1]).resolve()
sys.path.insert(0,str(folder))
import hw_workflow as hw
hw.replay_directory(folder)
contract=json.loads((folder/'contract.json').read_text())
table=hw.load_table(folder/'frozen.csv',contract)
frozen=json.loads((folder/'experiment.json').read_text())
result=hw.run_frozen(table,frozen,frozen['source'],frozen['config'],frozen['plan'])
Path(sys.argv[2]).write_text(json.dumps({'ids':result['test_ids'],'predictions':result['predictions']}))
"""
_capture_file=BRIDGE_DIR/'capture-receiver-vectors.py'
_capture_file.write_text(BRIDGE_CAPTURE_SOURCE)
_capture_out=(BRIDGE_DIR/'receiver-vectors.json').resolve()
import subprocess
_capture=subprocess.run([sys.executable,str(_capture_file.resolve()),str(bridge_received.resolve()),str(_capture_out)],
    cwd=bridge_received,capture_output=True,text=True,timeout=60)
assert _capture.returncode==0,_capture.stderr
bridge_receiver_vectors=json.loads(_capture_out.read_text())
# Draw the actual new-process receiver vectors against saved reference values.
with (bridge_received/'reference-predictions.csv').open() as _f:
    _ref = {r['run_id']:r for r in csv.DictReader(_f)}
bridge_differences = {}
for _name in ['baseline','candidate']:
    _saved = np.array([float(_ref[i][_name]) for i in bridge_receiver_vectors['ids']])
    _actual = np.array(bridge_receiver_vectors['predictions'][_name])
    np.testing.assert_allclose(_actual,_saved,atol=1e-8,rtol=1e-10)
    bridge_differences[_name] = _actual - _saved
with plt.rc_context(BRIDGE_PLOT_STYLE):
    _fig, _axes = plt.subplots(1,2,figsize=(10,4),layout='constrained')
for _ax, _name in zip(_axes,['baseline','candidate']):
    _ax.plot(np.arange(120), bridge_differences[_name], '.', ms=4)
    _ax.axhline(0, color='black', lw=1)
    _ax.set(title=_name, xlabel='Aligned test-ID position', ylabel='prediction difference', ylim=(-1e-8,1e-8))
    _ax.title.set_fontsize(14); _ax.xaxis.label.set_fontsize(11); _ax.yaxis.label.set_fontsize(11)
_fig.suptitle('Actual receiver differences by ID (not target error)',fontsize=13)
_fig.savefig(BRIDGE_DIR/'id-aligned-replay.png', dpi=140)
plt.show(); plt.close(_fig)
print('This is classroom self-replay. No other person has verified it.')


### Bridge 3 · Change one line and test a refusal (5 min)
First fill `rejection_prediction`. Change `BRIDGE_INVALID_UNIT` from `km/h` to another wrong declared airflow unit, such as `ft/s`, and rerun the next cell. Explain why merely changing a label does not perform a conversion. Require the specific unit-contract message and unchanged original bytes. The second branch intentionally has no package: it must export a **draft**, even though execution finishes.

In [ ]:
# Student code edit: change km/h to another incorrect unit, then rerun.
# Predict the rejecting mechanism first. Keep the original CSV and contract frozen.
BRIDGE_INVALID_UNIT = 'km/h'
assert BRIDGE_INVALID_UNIT != 'm/s', 'Choose one WRONG declared unit for this rejection test.'
def bridge_invalid_contract(contract):
    contract['units']['airflow_m_s'] = BRIDGE_INVALID_UNIT
_before_contract = (bridge_received/'contract.json').read_bytes()
_before_data = (bridge_received/'frozen.csv').read_bytes()
bridge_rejection = bridge_hw.negative_test(bridge_received, bridge_invalid_contract)
print(bridge_rejection)
assert bridge_rejection['expected_rejection']
assert bridge_rejection.get('message','').startswith('unit contract mismatch')
assert (bridge_received/'contract.json').read_bytes() == _before_contract
assert (bridge_received/'frozen.csv').read_bytes() == _before_data
# Also inspect the no-package branch: no exception is not proof of completion.
bridge_draft = bridge_hw.hw10_export(
    {'returncode':None,'status':'upload required'},
    {'status':'not implemented','expected_rejection':False},
    {k:'' for k in bridge_hw.REPORT_FIELDS}, '', 'self-replay', '',
    out_dir=BRIDGE_DIR/'missing-package-example')
assert bridge_draft['status'] == 'draft or unresolved replay failure'
print('No package + blank responses =>', bridge_draft['status'])


### Bridge 4 · Export and move to your own HW10 (5 min)
Update all four bridge responses with what you observed, then run the export cell. Download `week10-hw9-hw10-demo.zip` using the Colab Files sidebar under artifacts/week10-hw9-hw10-demo, or the Jupyter link. This is an example archive, **not** a Homework submission. Save your edited notebook separately.

For graded HW10, open the [real starter](https://colab.research.google.com/github/CHjonnyYu/MLES-Fall-2026/blob/main/notebooks/homework/hw10-thermal-workflow-starter.ipynb), upload a completed **student HW9** ZIP, implement your own negative test and record real peer actions or the stated self-replay fallback. Submit your executed HW10 notebook, evidence ZIP and one-page English explanation by **13 November 2026, 23:59 Asia/Taipei**. The midterm on 16 November remains written and oral only; there is no midterm project.

In [ ]:
# Create an example receiver receipt with honest self-replay labeling.
_br = bridge_receipt['result']
BRIDGE_RECEIVER_REPORT = {
 'change':'This classroom receiver inspected a supplied HW9-format demonstration package. The sender changed only Ridge regularization and retained its frozen comparison. I used the real HW10 replay functions in a newly extracted directory. I then changed one copied airflow unit declaration to '+BRIDGE_INVALID_UNIT+'. The original data and original contract remained unchanged.',
 'prediction':'Before execution, I expected both saved prediction vectors to reproduce after matching run IDs. I expected the wrong declared airflow unit to fail the input contract before model fitting. I did not infer success from opening the ZIP, seeing no uncaught exception or matching one rounded score. Those observations would not establish vector agreement.',
 'result':f'The actual child process returned {bridge_receipt["returncode"]}. It compared {_br["rows_compared"]} rows for both baseline and candidate. Maximum absolute differences were {_br["max_abs_difference"]}. The invalid contract produced the recorded message: '+bridge_rejection['message']+'. The separate missing-package example exported a draft, correctly showing that a completed code run is not a completed receiver submission.',
 'evidence':'The exported receiver ZIP preserves the actual process receipt, package checksum, environment, rejection outcome and attempt record. A measured chart shows differences at every aligned test position. The sender package preserves both target-error charts. These figures answer different questions. The example manifest explicitly identifies classroom demonstration material, and the real Homework starter refuses it as a student submission.',
 'limitations':'This is self-replay of a supplied classroom example, not a real classmate exchange. A fresh process does not establish independent human verification or a rebuilt environment. The correct course fallback is to state self-replay and leave teacher or TA verification pending. A real HW10 submission must use a completed student HW9 package, actual receiver actions, feedback on the author package and the student’s own explanation.'
}
bridge_receiver = bridge_hw.hw10_export(bridge_receipt, bridge_rejection,
    BRIDGE_RECEIVER_REPORT, BRIDGE_DISCLOSURE, 'self-replay',
    'Supplied classroom example operated by one person; no peer verification claimed.',
    history=[bridge_receipt], package_path=BRIDGE_PACKAGE, out_dir=BRIDGE_DIR/'receiver-exports')
assert bridge_receiver['status'] == 'self-replay; instructor/TA verification pending'
bridge_evidence = {
 'purpose':'Guided classroom demonstration; not a Homework submission',
 'rows_compared':_br['rows_compared'], 'replay':_br,
 'negative_test':bridge_rejection, 'no_package_status':bridge_draft['status'],
 'receiver_status':bridge_receiver['status'],
 'author_package_sha256':bridge_hw.digest(BRIDGE_PACKAGE.read_bytes()),
 'responses':bridge_responses,
 'blank_response_fields':[k for k,v in bridge_responses.items() if not str(v).strip()],
 'actual_invalid_unit':BRIDGE_INVALID_UNIT,
 'student_submission_route':'Use your own completed HW9 package in the separate HW10 starter.'
}
bridge_hw.write_json(BRIDGE_DIR/'bridge-evidence.json', bridge_evidence)
BRIDGE_EXPORT = BRIDGE_DIR/'week10-hw9-hw10-demo.zip'
with zipfile.ZipFile(BRIDGE_EXPORT,'w',zipfile.ZIP_DEFLATED) as _z:
    _z.write(BRIDGE_PACKAGE,'demo-hw09-evidence.zip')
    _z.write(bridge_receiver['zip'],'demo-hw10-evidence.zip')
    _z.write(BRIDGE_DIR/'bridge-evidence.json','bridge-evidence.json')
    _z.write(BRIDGE_DIR/'id-aligned-replay.png','id-aligned-replay.png')
    _z.write(BRIDGE_DIR/'receiver-vectors.json','receiver-vectors.json')
    _z.writestr('README.txt','GUIDED CLASSROOM EXAMPLE ONLY. Do not submit this ZIP or either nested ZIP as Homework. Use your own completed HW9 evidence ZIP in the real HW10 starter. Self-replay is not independent peer verification.\n')
print('Demo export:', BRIDGE_EXPORT)
print('Real HW10: https://mles-course-vgqg4abvka-de.a.run.app/notes/homework/hw10-thermal-handover.html')
print('Save your notebook separately. The eight-task classroom ZIP remains separate.')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(BRIDGE_EXPORT)))
except ImportError:
    pass


### Bridge summary
Preserve the experiment, inspect the correct package, compare every ID-aligned vector, test the declared failure mechanism, and label the verification honestly. Successful code execution, complete text fields and independent peer verification are three different claims. [HW10 requirements and rubric](https://mles-course-vgqg4abvka-de.a.run.app/notes/homework/hw10-thermal-handover.html).